# 0/1 Knapsack Problem using Dynamic Programming

## Introduction
The **0/1 Knapsack problem** is a classic optimization problem. We are given:

- A set of `n` items, each with a **weight** `w[i]` and a **value** `v[i]`
- A knapsack with a maximum **capacity** `cap`

The goal is to choose a subset of items so that the **total value is maximized** while the **total weight does not exceed the capacity**.

The "0/1" means each item is either taken **whole (1)** or **left out (0)**. Items cannot be split or reused, which is what separates this from the *Fractional Knapsack* problem (solved greedily).

## Approach
Greedy strategies such as "pick the highest value first" or "best value-to-weight ratio first" can fail for 0/1 Knapsack. We use **bottom-up dynamic programming** instead, which guarantees an optimal answer by building solutions from smaller subproblems.

This notebook:
1. Builds a DP table of best achievable values
2. Backtracks through the table to recover **which items** were chosen
3. Tests it on a sample input
4. Analyses time and space complexity

## Step 1: Import Libraries
We use **NumPy** to create and index the 2-D DP table efficiently. It is the only dependency.

In [37]:
import numpy as np

## Step 2: Knapsack Function

### Function signature
`Knapsack(w, v, cap)` takes:

| Parameter | Description |
|-----------|-------------|
| `w` | List/array of item weights |
| `v` | List/array of item values |
| `cap` | Maximum capacity of the knapsack |

It returns a tuple: `(maximum_value, list_of_selected_item_indices)`.

### Part A: Building the DP table
We create a table `dp` of size `(n+1) × (cap+1)` where:

> `dp[i][j]` = the maximum value achievable using only the **first `i` items** with a knapsack capacity of **`j`**.

Row `0` (no items) and column `0` (zero capacity) remain `0`.

For each item `i` and capacity `j`, there are two cases:

1. **Item does not fit** (`w[i-1] > j`): we cannot take it, so
   `dp[i][j] = dp[i-1][j]`
2. **Item fits** (`w[i-1] <= j`): we take the better of two options:
   - **Skip** the item: `dp[i-1][j]`
   - **Take** the item: `v[i-1] + dp[i-1][j - w[i-1]]`

   `dp[i][j] = max(skip, take)`

Note that `i-1` is used for `w` and `v` because the table is 1-indexed (row 0 is the empty case) while the arrays are 0-indexed.

### Part B: Backtracking to find the chosen items
The answer `dp[n][cap]` gives only the best value. To find the items, we walk backwards from `dp[n][cap]`:

- Start at `i = n`, `j = cap`
- If the item fits **and** `dp[i-1][j] < v[i-1] + dp[i-1][j - w[i-1]]`, taking item `i-1` improved the result, so we **record it** and reduce capacity: `j -= w[i-1]`
- Either way, move up one row: `i -= 1`
- Stop when `i` or `j` reaches `0`

The collected indices are reversed so they appear in ascending order. The returned indices are **0-based** positions in the input arrays.


In [38]:
def Knapsack(w,v,cap):
    # w   : List of item weights
    # v   : List of item values
    # cap : Maximum capacity of the knapsack
    n = len(w)
    dp = np.zeros((n+1,cap+1))
    for i in range(1,n+1):
        for j in range(1,cap+1):
            if(w[i-1] <= j):
                dp[i][j] = max(v[i-1] + dp[i-1][j-w[i-1]] , dp[i-1,j])
            else: dp[i][j] = dp[i-1,j]
    i = n
    j = cap
    ans =[]
    while(i > 0 and j > 0):
        if(j >= w[i-1] and dp[i-1,j] < v[i-1] + dp[i-1][j-w[i-1]]):
            ans.append(i-1)
            j -= w [i-1]    
        i -= 1
    ans.reverse()
    return dp[n][cap],ans
    

## Step 3: Define the Input
We set up a sample problem with **7 items**:

| Item index | 0 | 1 | 2 | 3 | 4 | 5 | 6 |
|-----------|---|---|---|---|---|---|---|
| **Weight** | 2 | 3 | 4 | 5 | 1 | 6 | 3 |
| **Value**  | 3 | 4 | 5 | 8 | 2 | 9 | 6 |

The knapsack **capacity is 12**.


In [39]:
w = np.array([2, 3, 4, 5, 1, 6, 3])       # Weights
v = np.array([3, 4, 5, 8, 2, 9, 6])       # Values
capacity = 12     

## Step 4: Run the Algorithm
We call `Knapsack(w, v, cap)` and unpack the result into:

- `a`: the maximum total value (profit) achievable
- `b`: the indices of the items placed in the knapsack

The result is printed in a readable format. Because the DP table is created with `np.zeros`, the maximum value is a float (e.g. `20.0`). It can be cast with `int(a)` if an integer is preferred.

In [40]:
a,b = Knapsack(w, v, capacity)
print(f"Max Profit: {a} , contents: {b}")   

Max Profit: 20.0 , contents: [1, 3, 4, 6]


## Results

**Output:** `Max Profit: 20.0 , contents: [1, 3, 4, 6]`

### Verification
The selected items (0-based indices) are:

| Item index | Weight | Value |
|-----------|--------|-------|
| 1 | 3 | 4 |
| 3 | 5 | 8 |
| 4 | 1 | 2 |
| 6 | 3 | 6 |
| **Total** | **12** | **20** |

- **Total weight = 3 + 5 + 1 + 3 = 12**, which equals the capacity, so the knapsack is exactly full and within the limit.
- **Total value = 4 + 8 + 2 + 6 = 20**, which is the maximum profit reported.

### Observations
- Item 5 has the highest raw value (9) but weighs 6. Picking it would leave less room for several high-ratio items, so the algorithm skips it, which shows why a greedy "highest value first" strategy can fail.
- The DP explores all valid combinations implicitly and guarantees the optimum.

## Complexity Analysis

Let **n** = number of items and **W** = knapsack capacity (`cap`).

### Time Complexity: O(n × W)
- Filling the DP table requires two nested loops (`n` rows × `W` columns), with constant-time work per cell. This costs **O(n × W)**.
- The backtracking loop moves up one row per iteration, so it takes at most **O(n)** steps.
- **Total: O(n × W) + O(n) = O(n × W)**

For this example: 7 × 12 = 84 cell computations.

> This is **pseudo-polynomial** time. It depends on the numeric value of `W`, not just the input size (the number of bits needed to represent `W`). The 0/1 Knapsack problem is NP-hard in general.

### Space Complexity: O(n × W)
- The `dp` table has `(n+1) × (W+1)` entries, giving **O(n × W)**.
- The `ans` list holds at most `n` indices, which is **O(n)**.
- **Total: O(n × W)**

### Possible Optimization
If only the **maximum value** is needed (not the chosen items), space can be reduced to **O(W)** using a single 1-D array iterated over capacities in **reverse**. Recovering the item list requires keeping the full table (or extra bookkeeping), so this implementation uses the 2-D version.

| Aspect | This implementation | Space-optimized (value only) |
|--------|--------------------|------------------------------|
| Time | O(n × W) | O(n × W) |
| Space | O(n × W) | O(W) |
| Returns chosen items | Yes | No |